**Large Language Models**

_Copyright Notice_

This notebook is available for educational purposes. You may not use or distribute this notebook for commercial purposes. You may make copies of this notebook and use or distribute it for educational purposes as long as you cite the author as the source of the notebook.

Jaeyoung Chun | School of Applied Artificial Intelligence | Handong Global University

# Retrieval-Augmented Generation

## Load Environment Variables

In [1]:
from dotenv import load_dotenv

In [2]:
# retrieve API key and other environment variables
load_dotenv()

True

## Loading

In [3]:
from langchain_community.document_loaders import PyPDFLoader

In [4]:
loader = PyPDFLoader("./data/세계의-홍수-전설.pdf")

In [5]:
pages = loader.load_and_split()

Ignoring wrong pointing object 6 0 (offset 0)
Ignoring wrong pointing object 8 0 (offset 0)
Ignoring wrong pointing object 10 0 (offset 0)
Ignoring wrong pointing object 12 0 (offset 0)


In [6]:
len(pages)

7

## Splitting

In [7]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [8]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=100
)

In [9]:
docs = text_splitter.split_documents(pages)

## Embeddings

In [10]:
from langchain_openai import OpenAIEmbeddings

In [11]:
embedding_model = OpenAIEmbeddings(model="text-embedding-3-large")

## Vector Store

In [12]:
from langchain_chroma import Chroma

In [13]:
db = Chroma.from_documents(docs, embedding_model)

## Retrieve

In [14]:
# MMR(Maximal Marginal Relevance)
retriever = db.as_retriever(
    search_type="mmr", search_kwargs={"k": 1, "fetch_k": 5, "lambda_mult": 0.8}
)

In [15]:
question = "노아의 홍수는 진짜 있었던 일인가요?"

In [16]:
context = retriever.invoke(question)

In [17]:
context

[Document(id='40c2f82c-c6c7-4a21-9694-539e37731fb3', metadata={'page': 6, 'moddate': "D:20241209114208Z00'00'", 'source': './data/세계의-홍수-전설.pdf', 'total_pages': 7, 'page_label': '7', 'author': 'Jaeyoung Chun', 'producer': 'macOS Version 14.5 (Build 23F79) Quartz PDFContext', 'creationdate': "D:20241209114208Z00'00'", 'creator': 'Word'}, page_content="지금까지 이러한 각지에 남아있는 동서양의 홍수전설의 유사점들을 미루어볼 때, 그 \n근저에는 인류의 기억에 깊은 각인을 남긴 어떤 실제적 사건이 존재했었다고 생각할 수 \n있지 않을까?  \n결론적으로 말한다면 노아와 그의 가족에서부터 시작하여 바벨탑의 혼란에 의해 노아의 \n후손들이 지구 전역으로 퍼져나가게 되고, 각 대륙과 섬들에 정착한 노아의 후손들이 각기 \n뿌리를 내린 지역의 문화나 풍토성에 영향을 받아 창세기 홍수와 방주의 기본골격은 어느 \n정도 유지한 채 지역과 민족성에 따라 조상으로부터 대대로 내려왔던 이야기들이 조금씩 \n원전(原典)에서 광이 윤색되고 변질되는 형태로 구비전승(具備傳承)되면서, 종국적으로 \n원래 성서에 기록되어 있는 홍수이야기와는 상당히 동떨어진 홍수전설로 남게된 것으로 \n추정된다. 예를 들면 태평양의 피지섬에선 창세기에 나오는 직사각형의 방주(Ark) 형태가 \n피지섬 원주민들에게 쉽게 이해될 수 있고 생활에 유용하게 쓰이고 가장 친숙한 \n카누(Canoe)의 형태로 변색되어 나타나고 있다는 점을 들 수 있다. \n이러한 홍수전설의 분포도가 EXPO 창조과학전시관 노아 홍수관에 전시되어 있다. 그리고 \n전시관 옆으로 통하는 영화관에선 대홍수의 장면을 실감 있는 스크린으로 보여주기도 

## Augment

In [18]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.prompts import HumanMessagePromptTemplate
from langchain_core.messages import SystemMessage

In [19]:
chat_template = ChatPromptTemplate.from_messages(
    [
        SystemMessage(
            content="당신은 지적설계를 지지하는 과학자입니다."
        ),
        HumanMessagePromptTemplate.from_template(
            """
            {question}
            아래의 문맥에 기반하여 답해주세요.
            {context}
            """
        )
    ]
)

In [20]:
message = chat_template.format_messages(
    question=question,
    context=context[0].page_content
)

## Generate

In [21]:
from langchain_openai import ChatOpenAI

### gpt-4o-mini

In [22]:
model = ChatOpenAI(
    model_name="gpt-4o-mini",
)

In [23]:
for chunk in model.stream(message):
    print(chunk.content, end="", flush=True)

노아의 홍수는 많은 문화에서 유사한 전설로 존재하며, 이러한 공통점은 인류의 역사에 깊은 인상을 남긴 실제 사건이 있었던 것으로 해석될 수 있습니다. 본 문맥에서는 각 대륙과 지역 문화에 따라 노아의 이야기와 방주에 대한 형상이 변형되고 전해져 내려왔다는 주장을 하고 있습니다. 노아와 그의 후손들이 퍼져나가면서 각 지역의 문화적 배경에 맞춰 홍수 전설이 형성되었다는 점에서 홍수의 실제 발생 가능성을 시사합니다.

예를 들어, 피지섬에서는 방주의 형태가 원주민들이 이해하기 쉽고 친숙한 카누 형태로 변형되어 나타나는 경우를 통해, 환경적 요인이 어떻게 전설에 영향을 미쳤는지를 보여줍니다. 이러한 설명은 노아의 홍수가 단순한 신화가 아니라, 역사적 사건이라는 논지를 강화하는 요소로 작용합니다.

결론적으로, 노아의 홍수가 실제로 존재했을 가능성에 대한 지지는 다양한 문화의 홍수 전설과 그 변형을 통해 나타나며, 이는 인류의 집단 기억 속에 뿌리내린 사건으로 볼 수 있다는 의견을 지지합니다.

### gpt-6-luna

⚠️ gpt-4o와 다르게 최근 GPT 모델들은 Persona 설정만으로는 세계관을 온전히 변경하기 힘듬.

In [24]:
model2 = ChatOpenAI(
    model_name="gpt-6-luna",
)

In [25]:
for chunk in model2.stream(message):
    print(chunk.content, end="", flush=True)

제시하신 문맥처럼 **노아 홍수를 실제 사건으로 해석하는 것은 가능합니다.** 여러 문화권에 홍수 이야기가 전해진다는 점은 인류가 큰 홍수 재난을 오래 기억해 왔다는 생각과 잘 맞습니다. 지적설계를 지지하는 관점에서도 이를 성경의 홍수 기록이 역사적 기억을 담았다는 해석의 한 근거로 볼 수 있습니다.

다만 홍수 전설들의 유사성만으로 **성경에 기록된 전 지구적 홍수가 실제로 일어났다고 과학적으로 입증할 수는 없습니다.** 비슷한 이야기는 실제 지역 홍수의 기억이 전해졌거나, 문화 간에 이야기가 퍼졌거나, 여러 사회가 큰 홍수를 각자 경험하면서 생겼을 수도 있습니다. 피지의 홍수 이야기에서 카누가 등장한다는 점도 흥미롭지만, 그것만으로 성경의 방주 이야기가 전승되어 변형된 것이라고 확정하기는 어렵습니다.

따라서 가장 신중한 답은 이렇습니다. **인류의 홍수 전승이 실제 재난의 기억을 담았을 가능성은 충분하지만, 그것이 곧 노아 시대의 전 지구적 홍수를 입증하는 것은 아닙니다.** 성경의 홍수를 역사적 사건으로 받아들이는 것은 신앙적·해석적 입장이며, 현재 주류 지질학의 증거는 최근 과거에 지구 전체를 덮은 단일 홍수가 있었다는 주장과는 일치하지 않습니다. 또한 홍수의 실제 범위를 메소포타미아 등의 대규모 지역 홍수로 해석하는 견해도 있습니다.